# Patent Strength Index (PSI) from CSV

This notebook reads patent metadata from an input CSV (`input_patents.csv`), computes the nine PSI components using the defined heuristics, combines them into a weighted Patent Strength Index (0–100), and writes the results to `psi_results.csv`.

Files included alongside this notebook:
- `input_patents.csv` — contains 3 sample US patents (you can replace with your own CSV)  
- `psi_results.csv` — produced by the notebook with per-patent component scores and final PSI

How to use:
1. Place this notebook and `input_patents.csv` in the same directory.  
2. Run all cells. The notebook will read the CSV, compute scores, and save `psi_results.csv`.

Heuristics and weights are documented in the code cell. Adjust `ROADMAP_KEYWORDS` or `PRODUCT_WINDOW_YEARS` as needed.

In [ ]:
import pandas as pd
from datetime import datetime
from typing import List

# Configuration
INPUT_CSV = 'input_patents.csv'
OUT_CSV = 'psi_results.csv'
PRODUCT_WINDOW_YEARS = 5
ROADMAP_KEYWORDS = ['ai','machine learning','sensor','security','battery','wireless','blockchain','cloud','autonomous']

WEIGHTS = {
    'claim_strength': 0.25,
    'prior_art_resilience': 0.15,
    'prosecution_history': 0.10,
    'citations_impact': 0.10,
    'family_coverage': 0.10,
    'remaining_term': 0.10,
    'commercial_fit': 0.10,
    'assignee_inventor_strength': 0.05,
    'enforceability_title': 0.05
}

def safe(d, key, default=None):
    try:
        return d[key]
    except Exception:
        return default

def score_claim_strength(row) -> float:
    num_claims = row.get('patent_num_claims', 0) or 0
    try:
        num_claims = int(num_claims)
    except Exception:
        num_claims = 0
    abstract = row.get('abstract','') or ''
    if num_claims <= 0:
        base = 10.0
    elif num_claims <= 5:
        base = 25.0 + (num_claims - 1) * 5.0
    elif num_claims <= 15:
        base = 50.0 + (num_claims - 6) * 3.5
    else:
        base = min(95.0, 85.0 + (num_claims - 16) * 0.5)
    if len(abstract) < 120:
        base *= 0.88
    return round(base,2)

def score_prior_art_resilience(row) -> float:
    refs = row.get('cited_patents_count', 0) or 0
    try:
        refs = int(refs)
    except Exception:
        refs = 0
    if refs == 0:
        return 90.0
    if refs <= 10:
        return 70.0
    if refs <= 50:
        return 50.0
    if refs <= 200:
        return 30.0
    return 10.0

def score_prosecution_history(row) -> float:
    app_date = row.get('application_date')
    grant_date = row.get('patent_date')
    if not app_date or not grant_date:
        return 50.0
    try:
        ad = datetime.fromisoformat(app_date)
        gd = datetime.fromisoformat(grant_date)
        months = max(0, (gd.year - ad.year) * 12 + (gd.month - ad.month))
    except Exception:
        return 50.0
    if months <= 12:
        return 85.0
    if months <= 36:
        return 60.0
    if months <= 72:
        return 40.0
    return 20.0

def score_citations_impact(row) -> float:
    fwd = row.get('citedby_patents_count', 0) or 0
    try:
        fwd = int(fwd)
    except Exception:
        fwd = 0
    if fwd == 0:
        return 10.0
    if fwd <= 5:
        return 30.0
    if fwd <= 20:
        return 55.0
    if fwd <= 100:
        return 75.0
    return 95.0

def score_family_coverage(row) -> float:
    # Placeholder: requires integration with family APIs for production use
    return 50.0

def score_remaining_term(row, product_window_years=PRODUCT_WINDOW_YEARS) -> float:
    app_date = row.get('application_date')
    if not app_date:
        return 50.0
    try:
        ad = datetime.fromisoformat(app_date)
    except Exception:
        return 50.0
    patent_term = 20.0
    now = datetime.utcnow()
    elapsed = (now - ad).days / 365.25
    remaining = max(0.0, patent_term - elapsed)
    if remaining >= product_window_years:
        return 100.0
    return round(100.0 * (remaining / product_window_years), 2)

def score_commercial_fit(row, roadmap_keywords: List[str]=ROADMAP_KEYWORDS) -> float:
    text = ((row.get('title') or '') + ' ' + (row.get('abstract') or '')).lower()
    if not text.strip():
        return 50.0
    matches = sum(1 for kw in roadmap_keywords if kw.lower() in text)
    if matches == 0:
        return 10.0
    if matches == 1:
        return 55.0
    if matches == 2:
        return 80.0
    return 95.0

def score_assignee_inventor_strength(row) -> float:
    assignee = row.get('assignee_organization','') or ''
    if not assignee:
        return 30.0
    names = assignee.lower()
    big = ['google','microsoft','ibm','intel','apple','samsung','amazon']
    if any(b in names for b in big):
        return 90.0
    return 60.0

def score_enforceability_title(row) -> float:
    assignee = row.get('assignee_organization')
    if not assignee:
        return 40.0
    return 80.0

def compute_all_scores(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for _, r in df.iterrows():
        row = r.to_dict()
        comps = {}
        comps['claim_strength'] = score_claim_strength(row)
        comps['prior_art_resilience'] = score_prior_art_resilience(row)
        comps['prosecution_history'] = score_prosecution_history(row)
        comps['citations_impact'] = score_citations_impact(row)
        comps['family_coverage'] = score_family_coverage(row)
        comps['remaining_term'] = score_remaining_term(row)
        comps['commercial_fit'] = score_commercial_fit(row)
        comps['assignee_inventor_strength'] = score_assignee_inventor_strength(row)
        comps['enforceability_title'] = score_enforceability_title(row)
        final = 0.0
        for k,w in WEIGHTS.items():
            final += comps.get(k,0.0) * w
        comps['psi_score'] = round(final,2)
        out = {
            'patent_number': row.get('patent_number'),
            'title': row.get('title') or row.get('patent_title'),
            'abstract': row.get('abstract'),
            'patent_date': row.get('patent_date'),
            'application_date': row.get('application_date')
        }
        out.update(comps)
        results.append(out)
    return pd.DataFrame(results)

if __name__ == '__main__':
    df_in = pd.read_csv(INPUT_CSV)
    df_out = compute_all_scores(df_in)
    df_out.to_csv(OUT_CSV, index=False)
    print(f'Wrote results to {OUT_CSV}')
    display(df_out)
